# 01 — Baseline Cyber Risk Loss Model

## What we're doing and why

Most organisations express cyber risk as a category — High, Medium, Low. That framing is useful for triage, but it breaks down when you need to make capital allocation decisions.

This notebook replaces the category with a **financial distribution**: a range of possible annual losses, each with an associated probability. The output is not a single number — it is a curve that shows:

- The **median** annual loss (what a typical year looks like)
- The **tail risk** (what a bad year looks like)
- The **probability of exceeding** any given loss threshold

### The model: FAIR-inspired ALE

```
ALE = TEF × Loss Magnitude
```

| Input | Meaning | Distribution |
|-------|---------|-------------|
| **TEF** — Threat Event Frequency | How many times per year a threat actor attempts to cause harm | Triangular (min, mode, max) |
| **Loss Magnitude** | Financial cost per successful event | Lognormal |

We run **10,000 Monte Carlo iterations** — sampling from both distributions simultaneously — to build up a picture of all the ways a year could unfold.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

np.random.seed(42)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Scenario: Ransomware against a mid-size enterprise

SIMULATIONS = 10_000

# TEF: triangular distribution (min, mode, max events/year)
TEF = (1, 3, 6)

# Loss Magnitude: lognormal — calibrated to a ~$6M median loss per event
LOSS_MEAN  = 14.5
LOSS_SIGMA = 0.7

In [ ]:
# ── Risk engine ───────────────────────────────────────────────────────────────

def simulate_ale(tef_range, loss_mean, loss_sigma, n=10_000):
    """Monte Carlo ALE: triangular TEF × lognormal Loss Magnitude."""
    tef  = np.random.triangular(*tef_range, n)
    loss = np.random.lognormal(loss_mean, loss_sigma, n)
    return tef * loss


def fmt(v):
    return f"${v/1e6:.2f}M" if v >= 1e6 else f"${v/1e3:.0f}K"


ale = simulate_ale(TEF, LOSS_MEAN, LOSS_SIGMA, SIMULATIONS)

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────

p50      = np.percentile(ale, 50)
p90      = np.percentile(ale, 90)
p95      = np.percentile(ale, 95)
prob_10m = np.mean(ale > 10_000_000)

print("=== BASELINE RISK — ANNUALIZED LOSS EXPOSURE ===")
print(f"Median (P50):       {fmt(p50)}")
print(f"90th Percentile:    {fmt(p90)}")
print(f"95th Percentile:    {fmt(p95)}")
print(f"P(loss > $10M):     {prob_10m:.1%}")

In [ ]:
# ── Loss distribution histogram ───────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(11, 5))

ax.hist(ale, bins=60, color="#A100FF", alpha=0.75, density=True)
ax.axvline(p50, color="white",   linestyle="--", linewidth=1.5, label=f"Median  {fmt(p50)}")
ax.axvline(p90, color="#FFA500", linestyle="--", linewidth=1.5, label=f"VaR-90  {fmt(p90)}")
ax.axvline(p95, color="#FF4B4B", linestyle=":",  linewidth=1.5, label=f"VaR-95  {fmt(p95)}")

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e6:.0f}M"))
ax.set_xlabel("Annual Loss", fontsize=11)
ax.set_ylabel("Probability Density", fontsize=11)
ax.set_title("Annualized Loss Exposure — Baseline Ransomware Scenario", fontsize=13, fontweight="bold")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# ── Loss exceedance curve ─────────────────────────────────────────────────────
# Answers: "What is the probability of losing MORE than $X in a given year?"

sorted_ale = np.sort(ale)[::-1]
exceedance = np.arange(1, len(sorted_ale) + 1) / len(sorted_ale)

fig2, ax2 = plt.subplots(figsize=(11, 5))

ax2.plot(sorted_ale, exceedance, color="#A100FF", linewidth=2)
ax2.fill_between(sorted_ale, exceedance, alpha=0.10, color="#A100FF")
ax2.axhline(0.10, color="#FFA500", linestyle=":", linewidth=1.2, label="10% exceedance — VaR-90")
ax2.axhline(0.05, color="#FF4B4B", linestyle=":", linewidth=1.2, label="5% exceedance — VaR-95")

ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e6:.0f}M"))
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax2.set_xlabel("Annual Loss", fontsize=11)
ax2.set_ylabel("Probability of Exceeding", fontsize=11)
ax2.set_title("Loss Exceedance Curve", fontsize=13, fontweight="bold")
ax2.legend()
fig2.tight_layout()
plt.show()

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────

print("\n=== FINAL SUMMARY: BASELINE RISK ===")
print(f"Median Loss:       {fmt(p50)}")
print(f"90th Percentile:   {fmt(p90)}")
print(f"95th Percentile:   {fmt(p95)}")
print(f"P(loss > $10M):    {prob_10m:.1%}")

## Key Takeaways

| Insight | Implication |
|---------|-------------|
| Median loss is significantly lower than tail risk | Planning to the average underestimates downside exposure |
| The distribution has a long right tail | Catastrophic outcomes are rare but financially material |
| The exceedance curve quantifies any threshold | Boards can ask *"what's our worst-case 90th percentile?"* and get a dollar answer |

**The single most important shift:** Move from asking *"what is our risk rating?"* to *"what loss are we 90% confident we won't exceed?"* That question has a dollar answer. Boards can act on dollar answers.

→ Continue to `02_scenario_analysis.ipynb` to compare this against other threat types.